# PART A - Character Tokenization

## E1

In [1]:
import re

In [27]:
text = "NLP is fun!"

characters = list(text)

print(f"\n{characters=}")

print("""
--- Q1 ---
""")
print(f"{len(characters)=}")

print("""
--- Q2 ---
""")
print(f"{[c for c in characters if c != " "]=}")
print(f"{len([c for c in characters if c != " "])=}")


characters=['N', 'L', 'P', ' ', 'i', 's', ' ', 'f', 'u', 'n', '!']

--- Q1 ---

len(characters)=11

--- Q2 ---

[c for c in characters if c != " "]=['N', 'L', 'P', 'i', 's', 'f', 'u', 'n', '!']
len([c for c in characters if c != " "])=9


In [7]:
"""
--- Q3 ---
"""
def char_frequency(text):
    characters = [c.lower() for c in re.findall(r"[a-zA-Z]", text)]
    char_counts = {char: 0 for char in set(characters)}
    for c in characters:
        char_counts[c] += 1
    return char_counts

In [28]:
print("""
--- Q3 ---
""")
text = "Natural language processing is fascinating!"
freq = sorted(char_frequency(text).items(), key=lambda kv: kv[1], reverse=True)
print(f"{freq[:3]=}")


--- Q3 ---

freq[:3]=[('a', 6), ('n', 5), ('g', 4)]


# PART B - Word Tokenization

In [2]:

import nltk
from nltk.tokenize import word_tokenize, TreebankWordTokenizer, WordPunctTokenizer
from nltk import pos_tag


## E2

In [29]:
text = "Natural language processing is interesting."

words = word_tokenize(text)

print("""
--- Q1 ---
""")
print(f"{words=}")
print(f"{len(words)=}")


--- Q1 ---

words=['Natural', 'language', 'processing', 'is', 'interesting', '.']
len(words)=6


In [4]:
print("""
--- Q2 ---
""")
def naive_tokenizer(text):
    return text.split(" ")
    
naive_tokenizer(text)


--- Q2 ---



['Well,', "that's...", 'surprising', '(and', 'a', 'bit', 'odd)!']

In [5]:
print("""
--- Q3 ---
""")
text = "Well, that's... surprising (and a bit odd)!"
print(f"{text=}")
print(f"{word_tokenize(text)=}")
print(f"{naive_tokenizer(text)=}")


--- Q3 ---

text="Well, that's... surprising (and a bit odd)!"
word_tokenize(text)=['Well', ',', 'that', "'s", '...', 'surprising', '(', 'and', 'a', 'bit', 'odd', ')', '!']
naive_tokenizer(text)=['Well,', "that's...", 'surprising', '(and', 'a', 'bit', 'odd)!']


# E3

In [37]:
print("""
--- Q1 ---
""")
text = "I don't know. It's useful, isn't it?"
print(f"{text=}")
print(f"{word_tokenize(text)=}") 


--- Q1 ---

text="I don't know. It's useful, isn't it?"
word_tokenize(text)=['I', 'do', "n't", 'know', '.', 'It', "'s", 'useful', ',', 'is', "n't", 'it', '?']


In [8]:
print("""
--- Q2 + Q3 ---
""")
treebank = TreebankWordTokenizer()
punctuation = WordPunctTokenizer()
text = "New York has twenty-one streets with state-of-the-art cooling standards. But then again, it is New York."
# Treebank + punkt data for sentence separation
print(f"{word_tokenize(text)=}")
print(f"{len(word_tokenize(text))=}")
# Treebank form without sentense separation
print(f"{treebank.tokenize(text)=}")
print(f"{len(treebank.tokenize(text))=}")
# just a regex \w+|[^\w\s]+
print(f"{punctuation.tokenize(text)=}")
# from nltk.tokenize import word_tokenize, TreebankWordTokenizer, WordPunctTokenizer


--- Q2 + Q3 ---

word_tokenize(text)=['New', 'York', 'has', 'twenty-one', 'streets', 'with', 'state-of-the-art', 'cooling', 'standards', '.', 'But', 'then', 'again', ',', 'it', 'is', 'New', 'York', '.']
len(word_tokenize(text))=19
treebank.tokenize(text)=['New', 'York', 'has', 'twenty-one', 'streets', 'with', 'state-of-the-art', 'cooling', 'standards.', 'But', 'then', 'again', ',', 'it', 'is', 'New', 'York', '.']
len(treebank.tokenize(text))=18
punctuation.tokenize(text)=['New', 'York', 'has', 'twenty', '-', 'one', 'streets', 'with', 'state', '-', 'of', '-', 'the', '-', 'art', 'cooling', 'standards', '.', 'But', 'then', 'again', ',', 'it', 'is', 'New', 'York', '.']


| | Sentence-aware | Contractions | Needs data download | Best for |
|---|---|---|---|---|
| `word_tokenize` | Yes | Treebank-style split | Yes (punkt) | General English text |
| `TreebankWordTokenizer` | No | Treebank-style split | No | Single sentences, speed, no dependencies |
| `WordPunctTokenizer` | No | Breaks at apostrophes | No | Simple, predictable splitting |



### A Word

A "word" has no single definition, because several reasonable ones disagree on the same string.

**Why it's ambiguous**

- **Orthographic:** a sequence of characters between spaces. This fails immediately for languages written without spaces (Chinese, Japanese, Thai) and for punctuation attached to words (`go.`, `"hello"`).
- **Syntactic/semantic:** a unit that carries meaning or plays a grammatical role. By this standard, `can't` is two words (`can` + `not`), `New York` is one, and `ice cream` is arguably one despite the space.
- **Morphological:** a form of a lexeme. `run`, `runs`, `ran` are different word forms but one lemma, so "how many words" depends on whether you count forms or lemmas.
- **Task-relative:** what counts as a word depends on what you do next. A parser, a search index, and a language model's vocabulary each want different units.

So there is no ground truth for the tokenizer to recover. Any choice is a convention, and downstream tools must agree with it.

**The trade-off: splitting versus keeping together**

Every tokenizer must decide how aggressively to split at internal punctuation, and both directions cost something.

*Contractions* (`can't`, `l'université`, `qu'il`, `don't`):
- **Split** (`ca` + `n't`, `l'` + `université`): the pieces map to real syntactic units (a verb and a negation, an article and a noun), which helps parsing and tagging, and it keeps the vocabulary smaller because `université` is shared across contexts. The cost is artificial fragments (`ca`, `'t`) that are not words on their own and can confuse text meant for humans or for string matching.
- **Keep whole:** tokens match what readers see, and search and display work naturally. The cost is that `l'université` and `université` become unrelated vocabulary entries, which inflates the vocabulary and hides the shared stem.

*Hyphenated compounds* (`state-of-the-art`, `well-known`, `peut-être`, `anti-intellectual`):
- **Split:** each part is a known word, so unseen compounds are still handled and vocabulary stays compact. The cost is lost meaning: `peut-être` ("maybe") is not `peut` + `être`, and `co-operate` or `e-mail` lose their identity.
- **Keep whole:** lexicalized compounds survive as units. The cost is a long tail of rare, one-off tokens (`twenty-three-year-old`) that blow up the vocabulary and make data sparse.

**The underlying tension**

It is a granularity trade-off between linguistic fidelity and statistical convenience. Fine-grained splitting gives smaller vocabularies and better coverage of unseen words, but breaks units that mean something together. Coarse tokens preserve meaning and readability but produce sparser data and more out-of-vocabulary items. No setting is right for all cases, since lexicalized forms (`peut-être`) and productive ones (`well-known`) look identical on the surface and need lexical knowledge, not a rule, to tell apart.

This is part of why modern models moved to subword tokenization (BPE, WordPiece), which sidesteps the question by not committing to "words" at all.

Since you work with francophone terminology, this matters directly: clitic contractions (`l'`, `d'`, `qu'`) and hyphenated forms (`vice-recteur`, `sous-directeur`, `est-ce que`) are exactly the cases where an English-oriented tokenizer will make the wrong call, and domain terms like academic titles are often the lexicalized compounds you'd want kept whole.

# PART C - Sentence Tokenization

In [10]:
from nltk.tokenize import sent_tokenize


## E4

In [50]:
print("""
--- Q1 ---
""")
text = "NLP is interesting. Students can practise it. The exercises are useful."
print(f"{text=}")
print(f"{sent_tokenize(text)=}") 


--- Q1 ---

text='NLP is interesting. Students can practise it. The exercises are useful.'
sent_tokenize(text)=['NLP is interesting.', 'Students can practise it.', 'The exercises are useful.']


In [52]:
print("""
--- Q2 ---
""")
text = "Is NLP hard? Not really! It just takes practice."
print(f"{text=}")
print(f"{sent_tokenize(text)=}") 


--- Q2 ---

text='Is NLP hard? Not really! It just takes practice.'
sent_tokenize(text)=['Is NLP hard?', 'Not really!', 'It just takes practice.']


In [28]:
print("""
--- Q3 ---
""")
text = "Smith gave a talk on NLP. It lasted 1.5 hrs. He arrived at 9 a.m."
print(f"{text=}")
print(f"{sent_tokenize(text)=}") 


--- Q3 ---

text='Smith gave a talk on NLP. It lasted 1.5 hrs. He arrived at 9 a.m.'
sent_tokenize(text)=['Smith gave a talk on NLP.', 'It lasted 1.5 hrs.', 'He arrived at 9 a.m.']


### Q4 - Heuristic: abbreviation lexicon plus a next-token capitalization check

Keep a list of known abbreviations (`Dr`, `Mr`, `a.m`, `p.m`, `etc`, `U.S`). When the tokenizer meets a period, it looks at the word just before it and decides in two steps:

1. **Is the preceding word a known abbreviation?** If not, treat the period as a sentence boundary (assuming a capitalized word follows). If yes, go to step 2.
2. **What follows?** Look at the next token. If it is lowercase, a digit, or another punctuation mark, the period was almost certainly part of the abbreviation, so there is no boundary. If it is capitalized, the case is ambiguous and a second rule decides:
   - If the abbreviation is a *title or prefix* type (`Dr.`, `Mr.`, `Prof.`), it is almost always followed by a name, so do not split.
   - If it is a *terminal-capable* type (`a.m.`, `etc.`, `U.S.`), it can legitimately end a sentence, so split only if the next word is a known sentence starter (`The`, `He`, `However`) rather than something like a proper noun.

**Why this works**

The lowercase-follows test resolves most cases cheaply (`at 9 a.m. tomorrow`). The title-versus-terminal distinction handles the hard case: `Dr. Smith` must not split, while `He arrived at 9 p.m. The meeting began.` should.

**Where it still fails**

An abbreviation that ends a sentence and is followed by a capitalized proper noun (`He lives in the U.S. Smith lives in France.`) is genuinely ambiguous, and so is any abbreviation missing from the list. This is the gap Punkt addresses by learning abbreviations and sentence starters statistically from a corpus, instead of relying on a hand-written list.

For your domain, the list would need French academic abbreviations (`Pr.`, `M.`, `Mme`, `Dr`, `Univ.`, `Dép.`), where the title-type rule would carry most of the weight.

# PART D - Tokenization Across Languages

## Task 1: Is `split()` reasonable?

| Language | Reasonable? | Justification |
|---|---|---|
| English | Mostly | Words are space-separated, but the final token comes out as `NLP.` with the period attached. |
| German | Mostly | Same punctuation problem (`Sprachmodell.`). The compound `Sprachmodell` also stays as one token, which may or may not be what you want (see Task 2). |
| Chinese | No | There are no spaces, so `split()` returns the whole sentence as one token. |
| Japanese | No | Same problem. Particles (は, を) and verb endings (ます) are also written directly against content words. |
| Arabic | Partly | Words are separated by spaces, so `split()` returns something, but function words attached to the following word (Task 4) are not separated, and punctuation and diacritics add noise. |

## Task 2: German compounds and vocabulary explosion

German forms new nouns by concatenating existing ones with no space: `Sprach` + `modell`, `Donaudampfschifffahrtsgesellschaft`, `Hochschulverwaltungssystem`. Compounding is productive, so the number of possible compounds is combinatorially large: with N nouns, there are on the order of N² two-part compounds, N³ three-part ones, and so on.

If the tokenizer keeps each compound whole:

- The vocabulary grows with every new compound, and most compounds are rare, so the data is very sparse.
- Many compounds appear only once in a corpus (hapax legomena), so a model sees too few examples to learn their meaning.
- At test time, unseen compounds become out-of-vocabulary, even though every component is known.
- Related items such as `Sprachmodell`, `Sprachverarbeitung` and `Sprachwissenschaft` look unrelated, hiding the shared `Sprach`.

Decompounding, or subword tokenization, fixes this by letting a small set of components cover a huge number of compounds. The risk is the same as with hyphenated forms in French: splitting a lexicalized compound whose meaning is not the sum of its parts.

## Task 3: Segmenting Chinese and Japanese without spaces

**Dictionary-based approach:** keep a lexicon of known words, then scan the text and choose the segmentation that best fits the dictionary. Two classic strategies are:

- **Maximum matching (greedy):** at each position, take the longest dictionary word that matches, then continue after it. For 我学习自然语言处理, this gives 我 / 学习 / 自然语言处理 (or 自然 / 语言 / 处理, depending on the dictionary).
- **Best-path search:** build all possible segmentations as a graph of dictionary matches and pick the one with the highest score. The score can be word frequencies or a language model, so that likelier word sequences win. This handles ambiguity better than greedy matching, since the longest match is not always right.

Dictionaries have limits: new words and names are missing, and some strings segment differently in different contexts. Modern tokenizers therefore combine a dictionary with statistical or neural sequence labeling, tagging each character as "begins a word", "inside a word" or "ends a word".

**For Japanese**, the mixed scripts help as a cue. A change from kanji to hiragana often signals a boundary (私|は, 言語|処理|を), but it is not enough on its own, since verbs mix kanji and hiragana (学びます), so a morphological analyzer with a lexicon and grammar rules is typically used.

## Questions

**1. Which languages use spaces?**
English, German and Arabic use spaces between words. Chinese and Japanese do not. Spaces only partly help, though: German packs several words into one compound, and Arabic attaches function words to the next word.

**2. Why is an English-style strategy unsuitable for Chinese and Japanese?**
English-style tokenization assumes that spaces and punctuation mark word boundaries, so it can split on whitespace and then fix up punctuation. Chinese and Japanese have no such markers, so that strategy produces one huge token per clause. Word boundaries must be inferred from lexical knowledge or statistics. There is also no agreed definition of a "word": 自然语言处理 can be one unit or three, depending on the application. Japanese adds particles and inflection fused into the text, and mixed scripts (kanji, hiragana, katakana, Latin).

**3. Arabic-specific difficulties**

- **Attached clitics:** conjunctions (و "and", ف "so"), prepositions (ب "with/in", ل "to", ك "like") and the article ال "the" are written as prefixes, and pronouns as suffixes. For instance, وبالكتاب is "and with the book" (و + ب + ال + كتاب). A whitespace tokenizer sees one token, so `كتاب`, `الكتاب` and `وبالكتاب` become separate vocabulary entries, which causes sparsity.
- **Ambiguous segmentation:** a leading letter may be a clitic or part of the word. وجد means "he found" (the و is part of the root), not "and" + جد. Deciding requires a lexicon or context.
- **Rich root-and-pattern morphology:** many forms derive from a three-consonant root (k-t-b gives كتاب, كاتب, مكتبة), so surface forms vary widely.
- **Optional diacritics:** short vowels are usually omitted, so many different words share one spelling. Where diacritics are present (as in the exercise's الطّبيعيّة), they must be handled or normalized, or the same word appears in several forms.
- **Orthographic variation:** letters like alef (أ, إ, آ, ا) and ta marbuta/ha (ة, ه) are often written inconsistently, and decorative elongation (tatweel) can appear, so normalization is needed.
- **Right-to-left script:** this matters for data handling, as the garbled input in this exercise shows.

# LAB 2 MINI CHALLENGE

In [58]:
text = "omgggg this lab is SO gooood!! can't wait for lab3 :) check https://nlp.univ.dz #NLP @prof_ahmed"
tweet = TweetTokenizer()

print(f"{text=}")
print(f"{text.split(" ")=}")
print(f"{tweet.tokenize(text)=}")

text="omgggg this lab is SO gooood!! can't wait for lab3 :) check https://nlp.univ.dz #NLP @prof_ahmed"
text.split(" ")=['omgggg', 'this', 'lab', 'is', 'SO', 'gooood!!', "can't", 'wait', 'for', 'lab3', ':)', 'check', 'https://nlp.univ.dz', '#NLP', '@prof_ahmed']
tweet.tokenize(text)=['omgggg', 'this', 'lab', 'is', 'SO', 'gooood', '!', '!', "can't", 'wait', 'for', 'lab', '3', ':)', 'check', 'https://nlp.univ.dz', '#NLP', '@prof_ahmed']


**Recommendation: `TweetTokenizer`.**

**Why `.split(" ")` falls short**

Social-media text is noisy, and a bare space split leaves most of that noise untouched:
- Punctuation stays glued to words (`great!!!`, `lol.`), so the same word appears as many different tokens.
- Emoticons and emoji next to words (`happy:)`, `love❤️`) merge with them.
- Splitting on a single space also yields empty strings when users type double spaces, and newlines or tabs are not treated as separators at all (`.split()` without an argument would fix that, but not the other problems).

**What `TweetTokenizer` does better**

- **Keeps social-media entities intact:** hashtags (`#NLP`), mentions (`@user`), URLs, and emoticons (`:-)`, `<3`) stay as single tokens instead of being broken apart.
- **Separates attached punctuation** from words, so `great!!!` becomes `great` plus the punctuation.
- **Normalizes optionally:** `preserve_case=False` lowercases text while keeping emoticons like `:D` intact. `reduce_len=True` shortens elongations (`soooooo` → `sooo`), and `strip_handles=True` removes `@mentions` when you don't want them.
- **Needs no data download**, and it is fast, so it suits large streams of text.

```python
from nltk.tokenize import TweetTokenizer
tk = TweetTokenizer(preserve_case=False, reduce_len=True)
tk.tokenize("Sooooo happy with #NLP class :) @prof https://example.com")
# ['sooo', 'happy', 'with', '#nlp', 'class', ':)', '@prof', 'https://example.com']
```

**Caveats**

- Its rules are tuned for English. For French or Arabic content it will not handle clitics (`l'université`, `وبالكتاب`) or code-switching well, so for your francophone and Arabic-script data you would add language-specific preprocessing on top.
- It does not normalize URLs, mentions, or numbers by itself. If your model should not memorize individual handles or links, replace them with placeholders (`<URL>`, `<USER>`) as a separate step.
- For modern neural models, you would use the model's own subword tokenizer instead, and `TweetTokenizer` would serve mainly as a cleaning step before it.